# morphological_quantification_2026-01-02 — 02_whole_morph_geometry

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 02 | Per-Z Whole-Morph Geometry Review

Review whole-morph geometry on every image and every z plane. This notebook stays fully per-z and does not choose a final z representation yet.


## Cell Guide

- `Setup`: resolve the project root, import helper code, and define output paths.
- `Load Candidate Outputs`: read the saved per-z masks and candidate table from `01`.
- `Representative Single-Z Backbone Review`: pick one representative z plane per file and review the current curved backbone on DAPI.
- `Build Per-Z Geometry`: compute centerline-aware geometry overlays and geometry rows for every image and z plane.
- `All-Z Axis Overlay Review`: overlay every per-z axis on one representative DAPI plane per file so stack-to-stack consistency is easy to judge visually.
- `Geometry Review Pages`: save image-first BF/DAPI/marker geometry pages for every image across z.
- `Next Step`: later notebooks can decide how to use different z planes for different downstream measurements.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from matplotlib.lines import Line2D
from IPython.display import Image, Markdown, display

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().resolve().parent
elif (Path.cwd() / "notebooks").exists():
    ROOT = Path.cwd().resolve()
else:
    raise RuntimeError("Run this notebook from the project root or the notebooks/ directory.")

SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import morphology_quantification_helpers as mqh


## Settings Notes

- This notebook does **not** choose a final z plane for any image.
- It works on every per-z mask generated in `01`.
- Files flagged `include_in_analysis = False` in `results/manifests/analysis_manifest.tsv` are filtered out here.
- The organoid backbone is treated as a tube-like centerline that stays inside the mask, rather than as a straight PCA axis.
- The centerline is estimated from the mask medial axis, pruned, and reduced to the longest internal path so later measurement-specific choices can stay flexible.
- Small magenta rings mark the original backbone endpoints before the straight tip continuation.


In [ ]:
ANALYSIS_MANIFEST_PATH = ROOT / "results" / "manifests" / "analysis_manifest.tsv"
CANDIDATE_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_z_candidates.tsv"
GEOMETRY_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_per_z_geometry.tsv"
CONSENSUS_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_consensus_geometry.tsv"
CONSENSUS_CURATION_PATH = ROOT / "results" / "annotations" / "whole_morph_axis_consensus_review.json"
QC_DIR = ROOT / "results" / "qc" / "whole_morph_per_z_geometry"
REPRESENTATIVE_QC_DIR = ROOT / "results" / "qc" / "whole_morph_backbone_debug"
AXIS_OVERLAY_QC_DIR = ROOT / "results" / "qc" / "whole_morph_all_z_axis_overlays"
CONSENSUS_QC_DIR = ROOT / "results" / "qc" / "whole_morph_consensus_axis_review"
MAX_INLINE_GEOMETRY_PAGES = 0
AXIS_OVERLAY_ROWS_PER_PAGE = 4
CONSENSUS_ROWS_PER_PAGE = 4

QC_DIR.mkdir(parents=True, exist_ok=True)
REPRESENTATIVE_QC_DIR.mkdir(parents=True, exist_ok=True)
AXIS_OVERLAY_QC_DIR.mkdir(parents=True, exist_ok=True)
CONSENSUS_QC_DIR.mkdir(parents=True, exist_ok=True)
CONSENSUS_CURATION_PATH.parent.mkdir(parents=True, exist_ok=True)


## Load Candidate Outputs


In [ ]:
manifest_df = pd.read_csv(ANALYSIS_MANIFEST_PATH, sep="\t")
included_file_paths = set(
    manifest_df.loc[
        manifest_df["include_in_analysis"].fillna(True).astype(bool),
        "file_path",
    ].astype(str)
)
candidate_df = pd.read_csv(CANDIDATE_TABLE_PATH, sep="\t")
candidate_df = candidate_df.loc[candidate_df["file_path"].astype(str).isin(included_file_paths)].copy()
candidate_df = candidate_df.sort_values(["file_id", "z_index"]).reset_index(drop=True)
print(f"Loaded {len(candidate_df)} per-z candidate rows across {candidate_df['file_id'].nunique()} files.")


## Representative Single-Z Backbone Review


In [ ]:
representative_df = (
    candidate_df.sort_values(
        ["file_id", "area_fraction", "z_index"],
        ascending=[True, False, True],
    )
    .groupby("file_id", sort=True, as_index=False)
    .head(1)
    .sort_values("file_id")
    .reset_index(drop=True)
)

if representative_df.empty:
    print("No representative z planes were found in the candidate table.")
else:
    n_items = len(representative_df)
    n_cols = 5
    n_rows = int(np.ceil(n_items / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.1 * n_cols, 3.2 * n_rows),
        constrained_layout=True,
    )
    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes[None, :]
    flat_axes = axes.ravel()

    for ax, row in zip(flat_axes, representative_df.itertuples(index=False)):
        plane = mqh.load_plane_channels(ROOT / str(row.file_path), z_index=int(row.z_index))
        mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)
        centerline = mqh.centerline_from_mask(mask)

        img = mqh.robust_rescale(plane["dapi"])
        ax.imshow(img, cmap="gray", vmin=0.0, vmax=1.0)
        mqh.plot_mask_outline(ax, mask, color="yellow", linewidth=1.0)
        mqh.plot_centerline_overlay(
            ax,
            centerline_xy=centerline["centerline_xy"],
            midpoint_xy=centerline["midpoint_xy"],
            endpoint_a_xy=centerline["endpoint_a_xy"],
            endpoint_b_xy=centerline["endpoint_b_xy"],
            base_endpoint_a_xy=centerline["base_endpoint_a_xy"],
            base_endpoint_b_xy=centerline["base_endpoint_b_xy"],
            posterior_click_xy=None,
            line_color="white",
            line_width=1.3,
        )
        ax.set_title(
            f"{int(row.file_id):02d} | z={int(row.z_index)}\n"
            f"DAPI | tort={float(centerline['tortuosity']):.2f}",
            fontsize=8,
        )
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in flat_axes[n_items:]:
        ax.axis("off")

    dapi_page = REPRESENTATIVE_QC_DIR / "representative_backbone_dapi.png"
    fig.savefig(dapi_page, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(dapi_page.relative_to(ROOT))
    display(Image(filename=str(dapi_page)))


## Build Per-Z Geometry


In [ ]:
geometry_rows = []
page_paths = []

for file_id, sub in candidate_df.groupby("file_id", sort=True):
    sub = sub.sort_values("z_index").reset_index(drop=True)
    first_row = sub.iloc[0]
    z_count = len(sub)

    fig, axes = plt.subplots(
        3,
        z_count,
        figsize=(max(2.8 * z_count, 8.5), 8.9),
        constrained_layout=True,
    )
    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes[:, None]

    for col_idx, row in enumerate(sub.itertuples(index=False)):
        plane = mqh.load_plane_channels(ROOT / str(row.file_path), z_index=int(row.z_index))
        mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)
        centerline = mqh.centerline_from_mask(mask)
        pixel_size_um = float(plane["stack"].scale_um.get("X", np.nan))
        centerline_length_um = float(centerline["length_px"] * pixel_size_um) if np.isfinite(pixel_size_um) else np.nan
        centerline_chord_length_um = (
            float(centerline["chord_length_px"] * pixel_size_um)
            if np.isfinite(pixel_size_um)
            else np.nan
        )

        geometry_rows.append(
            {
                "image_id": row.image_id,
                "cohort_id": row.cohort_id,
                "canonical_position": row.canonical_position,
                "file_id": int(row.file_id),
                "file_path": row.file_path,
                "acquisition_date": row.acquisition_date,
                "acquisition_batch_label": row.acquisition_batch_label,
                "z_index": int(row.z_index),
                "z_count": int(row.z_count),
                "mask_path": row.mask_path,
                "threshold_method": row.threshold_method,
                "threshold_scale": float(row.threshold_scale),
                "threshold_value": float(row.threshold_value),
                "raw_component_count": int(row.raw_component_count),
                "area_px": int(row.area_px),
                "area_fraction": float(row.area_fraction),
                "major_axis_length_um": float(row.major_axis_length_um),
                "minor_axis_length_um": float(row.minor_axis_length_um),
                "aspect_ratio": float(row.aspect_ratio),
                "eccentricity": float(row.eccentricity),
                "solidity": float(row.solidity),
                "extent": float(row.extent) if hasattr(row, "extent") else np.nan,
                "touches_border": bool(row.touches_border),
                "axis_mode": str(centerline["method"]),
                "centerline_length_px": float(centerline["length_px"]),
                "centerline_length_um": centerline_length_um,
                "centerline_chord_length_px": float(centerline["chord_length_px"]),
                "centerline_chord_length_um": centerline_chord_length_um,
                "centerline_tortuosity": float(centerline["tortuosity"]),
                "centerline_midpoint_x_px": float(centerline["midpoint_xy"][0]),
                "centerline_midpoint_y_px": float(centerline["midpoint_xy"][1]),
                "centerline_endpoint_a_x_px": float(centerline["endpoint_a_xy"][0]),
                "centerline_endpoint_a_y_px": float(centerline["endpoint_a_xy"][1]),
                "centerline_endpoint_b_x_px": float(centerline["endpoint_b_xy"][0]),
                "centerline_endpoint_b_y_px": float(centerline["endpoint_b_xy"][1]),
                "centerline_start_tangent_x": float(centerline["start_tangent_xy"][0]),
                "centerline_start_tangent_y": float(centerline["start_tangent_xy"][1]),
                "centerline_end_tangent_x": float(centerline["end_tangent_xy"][0]),
                "centerline_end_tangent_y": float(centerline["end_tangent_xy"][1]),
                "centerline_point_count": int(centerline["n_points"]),
            }
        )

        bf_display = (
            mqh.robust_rescale(plane["brightfield"])
            if plane["brightfield"] is not None
            else np.zeros_like(plane["dapi"], dtype=np.float32)
        )
        dapi_display = mqh.robust_rescale(plane["dapi"])
        overlay_display = np.asarray(plane["overlay_rgb"], dtype=np.float32)

        panel_specs = [
            (bf_display, "BF", "gray"),
            (dapi_display, "DAPI", "gray"),
            (overlay_display, "Markers", None),
        ]

        for row_idx, (img, label, cmap) in enumerate(panel_specs):
            ax = axes[row_idx, col_idx]
            if cmap is None:
                ax.imshow(img)
            else:
                ax.imshow(img, cmap=cmap, vmin=0.0, vmax=1.0)
            mqh.plot_mask_outline(ax, mask, color="yellow", linewidth=1.0)
            mqh.plot_centerline_overlay(
                ax,
                centerline_xy=centerline["centerline_xy"],
                midpoint_xy=centerline["midpoint_xy"],
                endpoint_a_xy=centerline["endpoint_a_xy"],
                endpoint_b_xy=centerline["endpoint_b_xy"],
                base_endpoint_a_xy=centerline["base_endpoint_a_xy"],
                base_endpoint_b_xy=centerline["base_endpoint_b_xy"],
                posterior_click_xy=None,
            )
            if row_idx == 0:
                ax.set_title(f"z={int(row.z_index)}", fontsize=9)
            if col_idx == 0:
                ax.set_ylabel(label, fontsize=9)
            if row_idx == 2:
                ax.text(
                    0.5,
                    -0.10,
                    f"area={float(row.area_fraction):.3f} | aspect={float(row.aspect_ratio):.2f} | tort={float(centerline['tortuosity']):.2f}",
                    transform=ax.transAxes,
                    ha="center",
                    va="top",
                    fontsize=7,
                )
            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle(
        f"{first_row.canonical_position} | file {int(first_row.file_id):02d} | {first_row.acquisition_date}",
        fontsize=12,
    )
    page_path = QC_DIR / f"{int(first_row.file_id):02d}_all_z_whole_morph_geometry.png"
    fig.savefig(page_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    page_paths.append(page_path)

geometry_df = pd.DataFrame(geometry_rows).sort_values(["file_id", "z_index"]).reset_index(drop=True)
geometry_df.to_csv(GEOMETRY_TABLE_PATH, sep="\t", index=False)
print(f"Wrote {len(geometry_df)} per-z geometry rows to {GEOMETRY_TABLE_PATH}")
print(f"Wrote {len(page_paths)} geometry review pages to {QC_DIR}")


## All-Z Axis Overlay Review


In [ ]:
file_ids = sorted(geometry_df["file_id"].unique())
overlay_pages = []

if not file_ids:
    print("No per-z geometry rows were available for axis overlay review.")
else:
    for page_idx, start_idx in enumerate(range(0, len(file_ids), AXIS_OVERLAY_ROWS_PER_PAGE), start=1):
        page_file_ids = file_ids[start_idx : start_idx + AXIS_OVERLAY_ROWS_PER_PAGE]
        n_rows = len(page_file_ids)
        fig, axes = plt.subplots(
            n_rows,
            2,
            figsize=(11.2, 4.0 * n_rows),
            gridspec_kw={"width_ratios": [1.0, 0.34]},
            constrained_layout=True,
        )
        axes = np.asarray(axes)
        if axes.ndim == 1:
            axes = axes[None, :]

        for row_idx, file_id in enumerate(page_file_ids):
            img_ax = axes[row_idx, 0]
            legend_ax = axes[row_idx, 1]
            sub = geometry_df[geometry_df["file_id"] == int(file_id)].sort_values("z_index").reset_index(drop=True)
            if sub.empty:
                img_ax.axis("off")
                legend_ax.axis("off")
                continue

            display_row = sub.iloc[len(sub) // 2]
            display_plane = mqh.load_plane_channels(
                ROOT / str(display_row.file_path),
                z_index=int(display_row.z_index),
            )
            display_mask = tifffile.imread(ROOT / str(display_row.mask_path)).astype(bool)
            dapi_display = mqh.robust_rescale(display_plane["dapi"])

            img_ax.imshow(dapi_display, cmap="gray", vmin=0.0, vmax=1.0)
            mqh.plot_mask_outline(img_ax, display_mask, color="yellow", linewidth=1.0)

            if len(sub) == 1:
                color_values = [plt.cm.turbo(0.5)]
            else:
                color_values = [plt.cm.turbo(x) for x in np.linspace(0.05, 0.95, len(sub))]

            legend_handles = []
            for color, row in zip(color_values, sub.itertuples(index=False)):
                mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)
                centerline = mqh.centerline_from_mask(mask)
                centerline_xy = np.asarray(centerline["centerline_xy"], dtype=np.float64)
                img_ax.plot(
                    centerline_xy[:, 0],
                    centerline_xy[:, 1],
                    color=color,
                    linewidth=1.8,
                    alpha=0.95,
                )
                legend_handles.append(
                    Line2D(
                        [0],
                        [0],
                        color=color,
                        linewidth=2.2,
                        label=f"z={int(row.z_index)}",
                    )
                )

            img_ax.set_title(
                f"{int(file_id):02d} | display z={int(display_row.z_index)} | all per-z axes",
                fontsize=10,
            )
            img_ax.set_xticks([])
            img_ax.set_yticks([])

            legend_ax.axis("off")
            legend_ax.legend(
                handles=legend_handles,
                loc="center left",
                frameon=False,
                fontsize=8,
                title="Axis Color",
                title_fontsize=8,
                ncol=1,
            )

        overlay_page = AXIS_OVERLAY_QC_DIR / f"all_z_axis_overlays_dapi_page{page_idx:02d}.png"
        fig.savefig(overlay_page, dpi=180, bbox_inches="tight")
        plt.close(fig)
        overlay_pages.append(overlay_page)

    for overlay_page in overlay_pages:
        print(overlay_page.relative_to(ROOT))
        display(Image(filename=str(overlay_page)))


## Manual Z Omission For Consensus Axis


In [ ]:
AXIS_Z_OMIT_BY_FILE = {}

AXIS_Z_KEEP_ONLY_BY_FILE = {}

curation_payload = {
    "axis_z_omit_by_file": {str(int(k)): [int(v) for v in values] for k, values in AXIS_Z_OMIT_BY_FILE.items()},
    "axis_z_keep_only_by_file": {
        str(int(k)): [int(v) for v in values] for k, values in AXIS_Z_KEEP_ONLY_BY_FILE.items()
    },
    "notes": "No manual z exclusions have been set for dataset 3 yet.",
}
CONSENSUS_CURATION_PATH.write_text(json.dumps(curation_payload, indent=2, sort_keys=True))
print(CONSENSUS_CURATION_PATH.relative_to(ROOT))


## Consensus Axis Review After Manual Omission


In [ ]:
consensus_rows = []
consensus_pages = []
file_ids = sorted(geometry_df["file_id"].unique())

if not file_ids:
    print("No per-z geometry rows were available for consensus-axis review.")
else:
    for page_idx, start_idx in enumerate(range(0, len(file_ids), CONSENSUS_ROWS_PER_PAGE), start=1):
        page_file_ids = file_ids[start_idx : start_idx + CONSENSUS_ROWS_PER_PAGE]
        n_rows = len(page_file_ids)
        fig, axes = plt.subplots(
            n_rows,
            2,
            figsize=(11.8, 4.15 * n_rows),
            gridspec_kw={"width_ratios": [1.0, 0.36]},
            constrained_layout=True,
        )
        axes = np.asarray(axes)
        if axes.ndim == 1:
            axes = axes[None, :]

        for row_idx, file_id in enumerate(page_file_ids):
            img_ax = axes[row_idx, 0]
            legend_ax = axes[row_idx, 1]
            sub_all = geometry_df[geometry_df["file_id"] == int(file_id)].sort_values("z_index").reset_index(drop=True)
            if sub_all.empty:
                img_ax.axis("off")
                legend_ax.axis("off")
                continue

            omit_set = {int(v) for v in AXIS_Z_OMIT_BY_FILE.get(int(file_id), [])}
            keep_only = [int(v) for v in AXIS_Z_KEEP_ONLY_BY_FILE.get(int(file_id), [])]
            if keep_only:
                sub_keep = sub_all[sub_all["z_index"].isin(keep_only)].copy()
                omitted_z = sorted(set(sub_all["z_index"].astype(int).tolist()) - set(sub_keep["z_index"].astype(int).tolist()))
            else:
                sub_keep = sub_all[~sub_all["z_index"].isin(sorted(omit_set))].copy()
                omitted_z = sorted(omit_set)

            if sub_keep.empty:
                sub_keep = (
                    sub_all.sort_values(["area_fraction", "z_index"], ascending=[False, True]).head(1).copy()
                )
                omitted_z = sorted(
                    set(sub_all["z_index"].astype(int).tolist()) - set(sub_keep["z_index"].astype(int).tolist())
                )

            sub_keep = sub_keep.sort_values("z_index").reset_index(drop=True)
            display_row = sub_keep.iloc[len(sub_keep) // 2]
            display_plane = mqh.load_plane_channels(
                ROOT / str(display_row.file_path),
                z_index=int(display_row.z_index),
            )
            display_mask = tifffile.imread(ROOT / str(display_row.mask_path)).astype(bool)
            dapi_display = mqh.robust_rescale(display_plane["dapi"])

            img_ax.imshow(dapi_display, cmap="gray", vmin=0.0, vmax=1.0)
            mqh.plot_mask_outline(img_ax, display_mask, color="yellow", linewidth=1.0)

            if len(sub_keep) == 1:
                color_values = [plt.cm.turbo(0.5)]
            else:
                color_values = [plt.cm.turbo(x) for x in np.linspace(0.05, 0.95, len(sub_keep))]

            retained_centerlines = []
            retained_z = []
            legend_handles = []
            for color, row in zip(color_values, sub_keep.itertuples(index=False)):
                mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)
                centerline = mqh.centerline_from_mask(mask)
                centerline_xy = np.asarray(centerline["centerline_xy"], dtype=np.float64)
                retained_centerlines.append(centerline_xy)
                retained_z.append(int(row.z_index))
                img_ax.plot(
                    centerline_xy[:, 0],
                    centerline_xy[:, 1],
                    color=color,
                    linewidth=1.6,
                    alpha=0.78,
                )
                legend_handles.append(
                    Line2D([0], [0], color=color, linewidth=2.2, label=f"z={int(row.z_index)}")
                )

            consensus_centerline = mqh.consensus_centerline_from_paths(
                retained_centerlines,
                reference_mask=display_mask,
                n_points=121,
                average_mode="mean",
            )
            mqh.plot_centerline_overlay(
                img_ax,
                centerline_xy=consensus_centerline["centerline_xy"],
                midpoint_xy=consensus_centerline["midpoint_xy"],
                endpoint_a_xy=consensus_centerline["endpoint_a_xy"],
                endpoint_b_xy=consensus_centerline["endpoint_b_xy"],
                base_endpoint_a_xy=None,
                base_endpoint_b_xy=None,
                posterior_click_xy=None,
                line_color="white",
                line_width=2.5,
            )
            legend_handles.append(Line2D([0], [0], color="white", linewidth=2.8, label="consensus"))

            img_ax.set_title(
                f"{int(file_id):02d} | display z={int(display_row.z_index)} | kept {len(retained_z)}/{len(sub_all)}",
                fontsize=10,
            )
            img_ax.set_xticks([])
            img_ax.set_yticks([])

            legend_ax.axis("off")
            legend_ax.legend(
                handles=legend_handles,
                loc="upper left",
                frameon=False,
                fontsize=8,
                title="Retained Z",
                title_fontsize=8,
                ncol=1,
            )
            keep_only_text = ", ".join(str(v) for v in keep_only) if keep_only else "None"
            omit_text = ", ".join(str(v) for v in omitted_z) if omitted_z else "None"
            legend_ax.text(
                0.02,
                0.20,
                f"keep only: {keep_only_text}\nomit: {omit_text}",
                transform=legend_ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
            )

            pixel_size_um = float(display_plane["stack"].scale_um.get("X", np.nan))
            consensus_rows.append(
                {
                    "image_id": display_row.image_id,
                    "cohort_id": display_row.cohort_id,
                    "canonical_position": display_row.canonical_position,
                    "file_id": int(file_id),
                    "file_path": display_row.file_path,
                    "display_z_index": int(display_row.z_index),
                    "retained_z_indices": ",".join(str(v) for v in retained_z),
                    "omitted_z_indices": ",".join(str(v) for v in omitted_z),
                    "keep_only_z_indices": ",".join(str(v) for v in keep_only),
                    "n_retained_z": int(len(retained_z)),
                    "n_total_z": int(len(sub_all)),
                    "consensus_axis_mode": str(consensus_centerline["method"]),
                    "consensus_centerline_length_px": float(consensus_centerline["length_px"]),
                    "consensus_centerline_length_um": float(consensus_centerline["length_px"] * pixel_size_um)
                    if np.isfinite(pixel_size_um)
                    else np.nan,
                    "consensus_centerline_chord_length_px": float(consensus_centerline["chord_length_px"]),
                    "consensus_centerline_chord_length_um": float(
                        consensus_centerline["chord_length_px"] * pixel_size_um
                    )
                    if np.isfinite(pixel_size_um)
                    else np.nan,
                    "consensus_centerline_tortuosity": float(consensus_centerline["tortuosity"]),
                    "consensus_centerline_xy_json": json.dumps(consensus_centerline["centerline_xy"].tolist()),
                    "consensus_midpoint_x_px": float(consensus_centerline["midpoint_xy"][0]),
                    "consensus_midpoint_y_px": float(consensus_centerline["midpoint_xy"][1]),
                    "consensus_endpoint_a_x_px": float(consensus_centerline["endpoint_a_xy"][0]),
                    "consensus_endpoint_a_y_px": float(consensus_centerline["endpoint_a_xy"][1]),
                    "consensus_endpoint_b_x_px": float(consensus_centerline["endpoint_b_xy"][0]),
                    "consensus_endpoint_b_y_px": float(consensus_centerline["endpoint_b_xy"][1]),
                }
            )

        consensus_page = CONSENSUS_QC_DIR / f"consensus_axis_review_dapi_page{page_idx:02d}.png"
        fig.savefig(consensus_page, dpi=180, bbox_inches="tight")
        plt.close(fig)
        consensus_pages.append(consensus_page)

    consensus_df = pd.DataFrame(consensus_rows).sort_values("file_id").reset_index(drop=True)
    consensus_df.to_csv(CONSENSUS_TABLE_PATH, sep="\t", index=False)
    print(CONSENSUS_TABLE_PATH.relative_to(ROOT))
    for consensus_page in consensus_pages:
        print(consensus_page.relative_to(ROOT))
        display(Image(filename=str(consensus_page)))


## Downstream Geometry Summary


In [ ]:
print(
    "Organoid-level whole-morph summary values are generated in 05f from "
    "05_posterior_oriented_consensus_axes.tsv so the geometry review and handoff "
    "reuse the same posterior-oriented axis object as downstream domain quantification."
)


## Geometry Review Pages


In [ ]:
if MAX_INLINE_GEOMETRY_PAGES <= 0:
    print("Inline across-z geometry pages suppressed. Per-z review PNGs are still saved to disk.")
else:
    preview_ids = sorted(geometry_df["file_id"].unique())[:MAX_INLINE_GEOMETRY_PAGES]
    for file_id in preview_ids:
        page_path = QC_DIR / f"{int(file_id):02d}_all_z_whole_morph_geometry.png"
        print(page_path.relative_to(ROOT))
        display(Image(filename=str(page_path)))


## Next Step

Later notebooks can decide which z planes to use for specific downstream measurements, posterior annotation, and marker-domain quantification. This notebook keeps geometry fully per-image and per-z.
